# Aula 3 — Comparando 2012 x 2026 e cruzando com afazeres domésticos

Nesta aula:
1. Abrir e juntar as duas **Tabelas 5** (composição da desocupação por sexo, 2012 e 2026)
2. Comparar ano a ano (quem ganhou/perdeu participação)
3. Cruzar com a **Tabela 1.1.1** do IBGE (horas semanais de afazeres domésticos, 2022)

In [1]:
import pandas as pd


## 1. Abrir a Tabela 5 de 2012

O CSV usa `;` como separador de coluna e `,` como separador decimal (padrão brasileiro) — por isso `sep=";"` e `decimal=","`. O `encoding="utf-8-sig"` remove o caractere invisível (BOM) que aparece no início do arquivo quando ele foi salvo pelo Excel.

In [2]:
path_2012 = "Tabela5-sem_emprego_2012.csv"

t2012 = pd.read_csv(path_2012, sep=";", decimal=",", encoding="utf-8-sig")


In [3]:
t2012


,Sigla,Código,Estado,Desocupados - homens (2012 T1),Desocupados - mulheres (2012 T1)
0,AC,12,Acre,45.3,54.7
1,AL,27,Alagoas,44.7,55.3
2,AM,13,Amazonas,47.5,52.5
3,AP,16,Amapá,48.1,51.9
4,BA,29,Bahia,43.8,56.2
5,CE,23,Ceará,51.1,48.9
6,DF,53,Distrito Federal,41.0,59.0
7,ES,32,Espírito Santo,46.4,53.6
8,GO,52,Goiás,41.0,59.0
9,MA,21,Maranhão,47.2,52.8


## 2. Abrir a Tabela 5 de 2026

Mesma lógica de leitura.

In [4]:
path_2026 = "Tabela5-sem_emprego_2026.csv"

t2026 = pd.read_csv(path_2026, sep=";", decimal=",", encoding="utf-8-sig")


In [5]:
t2026


,Sigla,Código,Estado,Desocupados - homens (2026 T1),Desocupados - mulheres (2026 T1)
0,AC,12,Acre,53.2,46.8
1,AL,27,Alagoas,50.7,49.3
2,AM,13,Amazonas,46.8,53.2
3,AP,16,Amapá,51.1,48.9
4,BA,29,Bahia,44.7,55.3
5,CE,23,Ceará,52.7,47.3
6,DF,53,Distrito Federal,47.9,52.1
7,ES,32,Espírito Santo,47.5,52.5
8,GO,52,Goiás,47.5,52.5
9,MA,21,Maranhão,49.1,50.9


## 3. Juntar as duas tabelas

Repare que `t2012` e `t2026` têm as mesmas colunas **Sigla**, **Código** e **Estado** — é isso que usamos como chave do `merge`. As colunas de valor (homens/mulheres) têm nomes diferentes em cada tabela (`... 2012 T1` vs `... 2026 T1`), então o `merge` não cria conflito: as duas ficam lado a lado.

In [6]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp


,Sigla,Código,Estado,Desocupados - homens (2012 T1),Desocupados - mulheres (2012 T1),Desocupados - homens (2026 T1),Desocupados - mulheres (2026 T1)
0,AC,12,Acre,45.3,54.7,53.2,46.8
1,AL,27,Alagoas,44.7,55.3,50.7,49.3
2,AM,13,Amazonas,47.5,52.5,46.8,53.2
3,AP,16,Amapá,48.1,51.9,51.1,48.9
4,BA,29,Bahia,43.8,56.2,44.7,55.3
5,CE,23,Ceará,51.1,48.9,52.7,47.3
6,DF,53,Distrito Federal,41.0,59.0,47.9,52.1
7,ES,32,Espírito Santo,46.4,53.6,47.5,52.5
8,GO,52,Goiás,41.0,59.0,47.5,52.5
9,MA,21,Maranhão,47.2,52.8,49.1,50.9


### Pergunta 1
Em quais estados a participação **das mulheres** entre os desocupados **mais aumentou** de 2012 para 2026?

*Dica: crie uma coluna com a diferença entre a coluna de mulheres de 2026 e a de 2012, e ordene.*

In [7]:
comp["variacao_mulheres"] = (
    comp["Desocupados - mulheres (2026 T1)"] - comp["Desocupados - mulheres (2012 T1)"]
)

maior_aumento = comp.sort_values("variacao_mulheres", ascending=False)[
    ["Estado", "variacao_mulheres"]
]

maior_aumento.head(5)


,Estado,variacao_mulheres
20,Rondônia,10.4
26,Tocantins,9.1
24,Sergipe,6.2
19,Rio Grande do Norte,2.8
11,Mato Grosso do Sul,1.9


### Pergunta 2
Em **quantos** estados a participação das mulheres **diminuiu** entre 2012 e 2026?

In [8]:
qtd_diminuiu = (comp["variacao_mulheres"] < 0).sum()

print(f"Participação das mulheres diminuiu em {qtd_diminuiu} de {len(comp)} estados.")


Participação das mulheres diminuiu em 20 de 27 estados.


## 4. Cruzar com a Tabela 1.1.1 (horas de afazeres domésticos)

Reaproveitamos a leitura da Aula 2 para carregar `indicador_1` a partir do `.xls` do IBGE (pulando título, cabeçalho em várias linhas e rodapé).

In [9]:
path_ibge = "Tabela_1_1_1.xls"

bruto = pd.read_excel(path_ibge, engine="xlrd", header=None)

indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)

for col in indicador_1.columns[1:]:
    indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")

indicador_1


,uf_regiao,total,total_branca,total_preta_parda,homem_branca,homem_preta_parda,mulher_branca,mulher_preta_parda
0,Brasil,16.985781,16.545930,17.333142,11.705136,11.746330,20.404106,22.037998
1,Norte,16.187037,15.925246,16.256032,11.588653,11.514646,19.295780,20.534188
2,Rondônia,17.014162,16.727733,17.101523,12.111127,12.544080,20.412887,21.061410
3,Acre,13.606903,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754
4,Amazonas,16.807290,15.987545,16.995493,12.233043,12.822584,18.764969,20.716986
5,Roraima,13.639257,13.651755,13.633606,10.814803,10.476601,16.135877,16.560244
6,Pará,16.474268,16.181627,16.560239,11.524306,11.026707,19.835110,21.591144
7,Amapá,13.586656,14.389180,13.415091,11.116821,10.749444,16.648579,15.908487
8,Tocantins,15.815781,15.701788,15.768728,10.733863,11.682616,19.663713,19.629399
9,Nordeste,18.499396,18.151700,18.610911,11.723907,11.848436,22.744666,23.716716


Os nomes de estado batem: `comp["Estado"]` (Tabela 5) usa o mesmo texto que `indicador_1["uf_regiao"]` (ex.: `"Minas Gerais"`, `"Amapá"`). Isso permite o `merge` mesmo com nomes de coluna diferentes, usando `left_on` / `right_on`.

**Atenção:** `indicador_1` também tem linhas de região (`"Norte"`, `"Nordeste"`...) e `"Brasil"`, que não existem em `comp` — por isso usamos `how="inner"`, que descarta automaticamente o que não casa.

In [10]:
final = comp.merge(indicador_1, left_on="Estado", right_on="uf_regiao", how="inner")

final[["Estado", "variacao_mulheres", "total"]]


,Estado,variacao_mulheres,total
0,Acre,-7.9,13.606903
1,Alagoas,-6.0,20.217129
2,Amazonas,0.7,16.807290
3,Amapá,-3.0,13.586656
4,Bahia,-0.9,17.849776
5,Ceará,-1.6,19.499293
6,Distrito Federal,-6.9,15.096351
7,Espírito Santo,-1.1,17.111106
8,Goiás,-6.5,15.186471
9,Maranhão,-1.9,16.669614


### Pergunta final
Existe relação entre o **aumento da desocupação feminina** (2012→2026) e as **horas semanais de afazeres domésticos**? Calcule a correlação entre `variacao_mulheres` e `total`.

In [11]:
correlacao = final["variacao_mulheres"].corr(final["total"])

print(f"Correlação entre variação da desocupação feminina e horas de afazeres domésticos: {correlacao:.2f}")


Correlação entre variação da desocupação feminina e horas de afazeres domésticos: 0.10


A correlação sai **próxima de zero** — ou seja, nesses dados, o quanto a participação feminina na desocupação mudou de 2012 a 2026 **não** parece estar associado ao número de horas semanais dedicadas a afazeres domésticos em cada estado. São duas dimensões distintas da desigualdade de gênero, que aqui não caminham juntas.